# ACCESS-IoT · Notebook 4 — additional baselines, ten seeds and held-out actors
Adds to the reference baselines of Notebook 2:
- **relationship-based access control (ReBAC)**, two more **insider-threat detectors** (per-user deviation, autoencoder),
  a **sequence model (GRU)** and an **off-policy contextual bandit (doubly robust)**;
- **10 model seeds** with 95% confidence intervals for every learned approach, bootstrap intervals for deterministic ones;
- an **actor-disjoint protocol for all seven threats and all tasks** (`actor_disjoint_all`): held-out malicious actors appear
  only in the test period, and feedback about excluded requests is withheld from mechanisms that learn from revealed outcomes.

Cells R0-R5 are the same code as Notebook 2 (settings, metrics, existing baselines). Cells S0-S4 are new.
The split is derived from released columns; outputs go to `reports/rev1/` (Table 6, Table D.1, Figs. 3 and 4 of the paper).
Runtime: a GPU and a high-RAM machine are recommended. Prerequisite: the data from Zenodo (see the README).

In [ ]:
# R0 — settings and shared helpers
import os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
# Data: download the Zenodo record (https://doi.org/10.5281/zenodo.24307413) into DATA_DIR and run unpack.py there,
# which gives DATA_DIR/builds/<build>/requests.parquet, ... . Outputs (tables, figures) go to REPORTS.
# On Colab: from google.colab import drive; drive.mount("/content/drive") and set DATA_DIR to the folder on your Drive.
DATA_DIR = Path(os.environ.get("ACCESS_IOT_DATA", "data"))
REPORTS = Path(os.environ.get("ACCESS_IOT_REPORTS", "reports"))
REL, REP = DATA_DIR / "builds", REPORTS
REP.mkdir(parents=True, exist_ok=True)
assert REL.exists(), f"no builds found in {REL}: download and unzip the Zenodo files first"
# main + difficulty variants (E5) + telemetry ablation (E6a: no telemetry, WUSTL-EHMS-2020 instead of CICIoMT2024)
RUN_BUILDS = [b for b in ["main", "sparse_feedback", "noisy_benign", "hard_fraud", "no_telemetry", "wustl_telemetry"]
              if (REL / b / "features/history.parquet").exists()]
SEEDS = [0, 1, 2]
N_JOBS = min(8, os.cpu_count() or 2)
import xgboost as xgb, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("builds:", RUN_BUILDS, "| device:", DEVICE)
P, L, D, B = 0, 1, 2, 3
DEC = np.array(["PERMIT", "LIMITED", "DENY", "BREAK_GLASS"])
ENC = {s: i for i, s in enumerate(DEC)}
VISIBLE_KINDS = ["request", "pip", "telemetry", "baseline"]
ID_COLS = {"RequestID", "Timestamp", "SessionID", "UserID", "PatientID", "DeviceID", "DelegationID", "DelegatorID", "DelegateeID"}
PRES = {"VitalSigns", "MedicalRecord", "Prescription", "Imaging"}

def load(b):
    d = pd.read_parquet(REL / b / "requests.parquet")
    for c in ("Timestamp", "FeedbackTime"): d[c] = pd.to_datetime(d[c])
    return d

def single_row(d, dic):
    cols = [c for c in dic.loc[dic.Visibility.isin(VISIBLE_KINDS), "Column"] if c in d and c not in ID_COLS]
    X = d[cols].copy()
    X["HourOfDay"] = d.Timestamp.dt.hour; X["HasDelegation"] = d.DelegationID.notna().astype(int)
    X["DeviceDeptMatchesUser"] = (d.DeviceDepartment == d.UserDepartment).astype(int)
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]): X[c] = X[c].astype(object).fillna("NA").astype(str).astype("category")
    return X

class Ctx:
    """Everything a baseline may use for one build (oracle columns are kept apart and used only for evaluation)."""
    def __init__(self, b):
        self.b = b; self.d = load(b); d = self.d
        self.dic = pd.read_csv(REL / b / "data_dictionary.csv"); self.pol = json.load(open(REL / b / "policy.json"))
        self.mask = np.load(REL / b / "views/rl_admissible_mask.npy")
        self.split = d.Split.to_numpy(); self.tr, self.va, self.te = (self.split == "train"), (self.split == "val"), (self.split == "test")
        self.X1 = single_row(d, self.dic)
        H = pd.read_parquet(REL / b / "features/history.parquet"); H.index = self.X1.index; self.H = H
        self.X2 = pd.concat([self.X1, H], axis=1)
        self.ev = pd.read_parquet(REL / b / "views/feedback_events.parquet")
        self.opt = d.OptimalDecision.map(ENC).to_numpy(); self.mal = d.IsMalicious.to_numpy() == 1
        self.grant = (d.RequestType == "DelegationGrant").to_numpy(); self.mis = (d.DelegationOutcome == "Misused").to_numpy()
        self.underdel = d.DelegationID.notna().to_numpy() & ~self.grant
        self.zeta = 1 + d.ResourceSensitivity.to_numpy(float) / 4
        self.day = ((d.Timestamp - d.Timestamp.min().normalize()).dt.days).to_numpy()

In [ ]:
# R1 — metrics for T1-T3 (and per-threat malicious-grant rate for E7)
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

def t1_metrics(a, c, rows):
    a, o, m = a[rows], c.opt[rows], c.mal[rows]; g = np.isin(a, [P, L, B])
    gr, mis, ud = c.grant[rows], c.mis[rows], c.underdel[rows]
    bgctx = c.mask[rows, 3] & (c.d.EmergencyFlag.to_numpy()[rows] == 1)
    r = {"Accuracy": (a == o).mean(), "MacroF1": f1_score(o, a, labels=[P, L, D, B], average="macro", zero_division=0),
         "MGR": g[m].mean(), "LDR": (a[o == P] == D).mean(), "EDR": (a[o == B] == D).mean(),
         "BG-Mis": np.isin(a[bgctx & m], [B, L]).mean() if (bgctx & m).any() else np.nan,
         "DG-Mis": np.isin(a[gr & mis], [P, L]).mean() if (gr & mis).any() else np.nan,
         "DG-CleanRefused": (a[gr & ~mis] == D).mean() if (gr & ~mis).any() else np.nan,
         "DU-Exp": g[ud & m].mean() if (ud & m).any() else np.nan, "DU-Den": (a[ud & ~m & (o != D)] == D).mean(),
         "Envelope violations": int((~c.mask[rows][np.arange(rows.sum()), a]).sum()),
         "Avg reward": immediate_reward(a, o, m, c.zeta[rows]).mean()}
    return {k: round(float(v), 4) for k, v in r.items()}

def per_threat(a, c, rows):
    th = c.d.ThreatType.to_numpy()[rows]; g = np.isin(a[rows], [P, L, B]); m = c.mal[rows]
    return {t: round(float(g[m & (th == t)].mean()), 4) for t in sorted(set(th[m]))}

def t2_metrics(score, c, rows):
    """Break-glass adjudication: emergency claims in break-glass contexts; genuine (oracle BREAK_GLASS) vs fraudulent (malicious)."""
    ctx = rows & c.mask[:, 3] & (c.d.EmergencyFlag.to_numpy() == 1)
    gen, fra = ctx & (c.opt == B), ctx & c.mal
    y = np.r_[np.ones(gen.sum()), np.zeros(fra.sum())]; s = np.r_[score[gen], score[fra]]
    if len(np.unique(y)) < 2: return {}
    thr = np.quantile(score[gen], 0.01)                      # honour 99% of genuine emergencies
    return {"T2 AUROC": round(roc_auc_score(y, s), 4), "T2 fraud granted @99% honoured": round(float((score[fra] >= thr).mean()), 4)}

def t3_metrics(risk, c, rows):
    """Delegation risk at grant time: misused (positive) vs clean grants."""
    g = rows & c.grant
    y = c.mis[g].astype(int)
    if y.min() == y.max(): return {}
    return {"T3 AUROC": round(roc_auc_score(y, risk[g]), 4), "T3 AUPRC": round(average_precision_score(y, risk[g]), 4),
            "T3 base rate": round(float(y.mean()), 4)}

def immediate_reward(a, o, mal, z):
    """reward of the benchmark's RL view (rl_env_spec.json); also the cost-sensitive objective used to tune every threshold"""
    r = np.where(a == o, 1.0, 0.0)
    r = np.where((a == L) & np.isin(o, [P, B]), 0.5, r)
    r = np.where(np.isin(a, [P, B]) & (o == L), -z, r)
    r = np.where(np.isin(a, [P, B]) & (o == D), np.where(mal, -5 * z, -2 * z), r)
    r = np.where((a == L) & (o == D), np.where(mal, -2.5 * z, -z), r)
    r = np.where((a == D) & np.isin(o, [P, L]), -1.0, r)
    r = np.where((a == D) & (o == B), -4.0, r)
    return r

def val_reward(a, c):
    v = c.va; return float(immediate_reward(a[v], c.opt[v], c.mal[v], c.zeta[v]).mean())

def tune(fn, grid, c):
    """pick the parameter tuple with the best average reward on the validation period"""
    best = None
    for g in grid:
        a = fn(*g); f = val_reward(a, c)
        if best is None or f > best[0]: best = (f, g, a)
    return best[1], best[2]

In [ ]:
# R2 — rule-based and policy-based approaches (no learning from labels)
def static_abac(c): return np.where(c.mask[:, 0], P, D)
def static_bg(c): return np.where(c.mask[:, 0], P, np.where(c.mask[:, 3], B, D))

def rbac(c):
    d = c.d; perm = set(map(tuple, c.pol["permissions"]))
    ok = np.array([(r, s, a) in perm for r, s, a in zip(d.UserRole, d.ResourceType, d.RequestedAction)])
    return np.where(c.grant, np.where(c.mask[:, 0], P, D), np.where(ok, P, D))         # roles only: no relationship check

def mined_abac(c):
    """ABAC policy mining: an interpretable decision tree learned from the logged (static) decisions of the training period."""
    from sklearn.tree import DecisionTreeClassifier
    d = c.d
    F = pd.get_dummies(d[["UserRole", "ResourceType", "RequestedAction", "RequestType", "DelegationScope"]].astype(str), dtype=np.uint8)
    F = pd.concat([F, d[["OnCareTeam", "SameDepartment", "PatientResearchConsent", "DelegationDepth", "RootDelegatorMaxDepth"]].reset_index(drop=True),
                   pd.DataFrame({"DeviceDeptMatch": (d.DeviceDepartment == d.UserDepartment).astype(int).to_numpy(),
                                 "HasDelegation": d.DelegationID.notna().astype(int).to_numpy(),
                                 "DepthWithinLimit": (d.DelegationDepth <= d.RootDelegatorMaxDepth).astype(int).to_numpy()})], axis=1)
    tree = DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=0).fit(F[c.tr], (d.ABACDecision[c.tr] == "Permit"))
    perm = tree.predict(F).astype(bool)
    fidelity = float((perm[c.te] == (d.ABACDecision.to_numpy()[c.te] == "Permit")).mean())
    return np.where(perm, P, D), {"mined rules (leaves)": int(tree.get_n_leaves()), "fidelity to logged policy (test)": round(fidelity, 4)}

def zero_trust(c):
    """Per-request re-evaluation of posture, network, authentication and context (fixed rules, no learning)."""
    d = c.d
    q = np.quantile(d.NetAnomaly[c.tr], 0.95) if "NetAnomaly" in d else None
    posture = (d.FirmwareAttestation == "Pass") & (d.DeviceAuthFailures24h == 0) & ((d.NetAnomaly <= q) if q is not None else True)
    network = (d.SourceNetwork != "External") | (d.AccessChannel == "Hospital Workstation")
    auth = d.LoginFailedAttempts == 0
    context = (d.WithinShift == 1) | (d.EmergencyFlag == 1)
    score = (posture.astype(int) + network.astype(int) + auth.astype(int) + context.astype(int)).to_numpy()
    a = np.where(c.mask[:, 0] & (score == 4), P, np.where(c.mask[:, 0] & (score == 3), L, D))
    return np.where(c.mask[:, 3] & (posture & auth).to_numpy(), B, a)

def risk_adaptive(c):
    """Risk = 0.5 anomaly + 0.25 sensitivity + 0.25 context; anomaly from an Isolation Forest fitted on training traffic (no labels)."""
    from sklearn.ensemble import IsolationForest
    num = c.X2.select_dtypes(include=[np.number]).fillna(0).to_numpy(np.float32)
    mu, sd = num[c.tr].mean(0), num[c.tr].std(0) + 1e-6; Z = (num - mu) / sd
    iso = IsolationForest(n_estimators=200, max_samples=8192, random_state=0, n_jobs=-1).fit(Z[c.tr])
    an = -iso.score_samples(Z); lo, hi = np.quantile(an[c.tr], [0.01, 0.99]); an = np.clip((an - lo) / (hi - lo), 0, 1)
    d = c.d
    ctx = ((1 - d.WithinShift) + (d.SourceNetwork == "External") + c.H.u_new_device.to_numpy()).to_numpy() / 3
    risk = 0.5 * an + 0.25 * (d.ResourceSensitivity.to_numpy() - 2) / 2 + 0.25 * ctx
    def pol(t1, t2, tb):
        a = np.where(c.mask[:, 0] & (risk < t1), P, np.where(c.mask[:, 0] & (risk < t2), L, D))
        return np.where(c.mask[:, 3] & (risk < tb), B, a)
    qs = np.quantile(risk[c.tr], [0.5, 0.7, 0.8, 0.9, 0.95, 0.98])
    grid = [(t1, t2, tb) for t1 in qs for t2 in qs if t2 >= t1 for tb in qs]
    g, a = tune(pol, grid, c)
    return a, risk, an

def isolation_insider(c, an):
    """Unsupervised insider detection on behaviour history: refuse the most anomalous requests."""
    def pol(qp, qb):
        a = np.where(c.mask[:, 0] & (an < qp), P, D)
        return np.where(c.mask[:, 3] & (an < qb), B, a)
    qs = np.quantile(an[c.tr], [0.8, 0.9, 0.95, 0.98, 0.99, 1.0])
    g, a = tune(pol, [(x, y) for x in qs for y in qs], c)
    return a

In [ ]:
# R3 — approaches that learn from delayed, action-dependent feedback (T4): CapBAC with revocation, trust-based (beta reputation)
def applies(kind_applies, a_src):
    return np.select([kind_applies == "always", kind_applies == "break_glass_granted", kind_applies == "delegation_approved"],
                     [np.ones(len(a_src), bool), np.isin(a_src, [B, L]), np.isin(a_src, [P, L])], np.isin(a_src, [P, L, B]))

def capbac(c, revoke):
    """Capability tokens: role rights (base) and delegated rights within scope, validity window and depth limit; no relationship check.
    With revoke=True, a principal's delegated capabilities are revoked once misuse evidence about it is revealed - but only
    evidence about requests the mechanism actually granted is ever revealed (replayed chronologically, day by day)."""
    d = c.d; perm = set(map(tuple, c.pol["permissions"]))
    role_ok = np.array([(r, s, a) in perm for r, s, a in zip(d.UserRole, d.ResourceType, d.RequestedAction)])
    deleg_ok = d.DelegationID.notna().to_numpy() & (d.CapabilityCheck == "Pass").to_numpy() & \
        (d.DelegationDepth <= d.RootDelegatorMaxDepth).to_numpy()
    own_ok = (d.UserRole != "Patient").to_numpy() | c.mask[:, 0]
    cap = np.where(c.grant, deleg_ok, (role_ok | deleg_ok) & own_ok)
    a = np.where(cap, P, D)
    if not revoke: return a
    # revocation acts on the principal that misused a delegated capability: its delegated tokens stop working, it can no
    # longer receive or re-issue delegations (tokens expire within hours, so revoking only the misused token would be too late)
    deleg = pd.read_parquet(REL / c.b / "delegations.parquet")[["DelegationID", "DelegateeID"]]
    holder_of = dict(zip(deleg.DelegationID, deleg.DelegateeID))
    rid2row = pd.Series(np.arange(len(d)), index=d.RequestID.to_numpy())
    ev = c.ev[c.ev.FeedbackLabel.isin(["Malicious", "DelegationMisused"]) & c.ev.DelegationID.notna()].copy()
    ev["row"] = rid2row[ev.RequestID.to_numpy()].to_numpy()
    ev["holder"] = np.where(ev.FeedbackLabel == "DelegationMisused", ev.DelegationID.map(holder_of), d.UserID.to_numpy()[ev.row.to_numpy()])
    ev["eday"] = ((ev.FeedbackTime - d.Timestamp.min().normalize()).dt.days).to_numpy()
    user, dlg_user = d.UserID.to_numpy(object), d.DelegateeID.to_numpy(object)
    uses_delegation = d.DelegationID.notna().to_numpy() & ~c.grant
    revoked = set(); days = np.unique(c.day); by_day = {k: np.flatnonzero(c.day == k) for k in days}; ev_by_day = ev.groupby("eday")
    for k in days:
        idx = by_day[k]
        if revoked:
            u, v = pd.Series(user[idx]).isin(revoked).to_numpy(), pd.Series(dlg_user[idx]).isin(revoked).to_numpy()
            hit = ((uses_delegation[idx] | c.grant[idx]) & u) | (c.grant[idx] & v)
            a[idx[hit]] = D
        if k in ev_by_day.groups:                            # evidence revealed today takes effect from tomorrow
            e = ev_by_day.get_group(k)
            ok = applies(np.where(e.FeedbackLabel == "DelegationMisused", "delegation_approved", "request_granted"), a[e.row.to_numpy()])
            revoked.update(e.holder[ok].dropna())
    return a

def trust_beta(c):
    """Trust-based access control: beta reputation (L+1)/(L+M+2) of the requester from revealed outcomes, times device reputation
    (0 once a device compromise is revealed). Outcomes reach the store only if the mechanism's own decision exposed them."""
    d = c.d; rid2row = pd.Series(np.arange(len(d)), index=d.RequestID.to_numpy())
    ev = c.ev.copy(); ev["row"] = rid2row[ev.RequestID.to_numpy()].to_numpy()
    ev["eday"] = ((ev.FeedbackTime - d.Timestamp.min().normalize()).dt.days).to_numpy()
    uid = pd.factorize(d.UserID)[0]; did = pd.factorize(d.DeviceID)[0]; nU, nD = uid.max() + 1, did.max() + 1
    days = np.unique(c.day); by_day = {k: np.flatnonzero(c.day == k) for k in days}; evd = {k: g for k, g in ev.groupby("eday")}
    last_val_day = c.day[c.va].max()
    def replay(tp, tb, upto=None):
        Lc, Mc, Dc = np.zeros(nU), np.zeros(nU), np.zeros(nD); a = np.full(len(d), D); T = np.ones(len(d))
        for k in (days if upto is None else days[days <= upto]):
            idx = by_day[k]
            T[idx] = (Lc[uid[idx]] + 1) / (Lc[uid[idx]] + Mc[uid[idx]] + 2) * (Dc[did[idx]] == 0)
            a[idx] = np.where(c.mask[idx, 0] & (T[idx] >= tp), P, np.where(c.mask[idx, 3] & (T[idx] >= tb), B, D))
            if k in evd:
                e = evd[k]; ok = applies(e.AppliesIf.to_numpy(), a[e.row.to_numpy()]); e = e[ok]
                r = e.row.to_numpy()
                np.add.at(Lc, uid[r[e.FeedbackLabel.to_numpy() == "Legitimate"]], 1)
                np.add.at(Mc, uid[r[e.FeedbackLabel.isin(["Malicious", "DelegationMisused"]).to_numpy()]], 1)
                np.add.at(Dc, did[r[(e.Kind == "device_detect").to_numpy()]], 1)
        return a, T
    # coarse-to-fine search on validation reward (64 + <=24 replays instead of 225); tuning replays stop after the validation period
    seen, best = set(), None
    def search(grid):
        nonlocal best
        for g in grid:
            if g in seen: continue
            seen.add(g); a, _ = replay(*g, upto=last_val_day); f = val_reward(a, c)
            if best is None or f > best[0]: best = (f, g)
    coarse = np.round(np.arange(0.05, 0.751, 0.1), 3)
    search([(tp, tb) for tp in coarse for tb in coarse])
    near = lambda v: [x for x in np.round(v + np.array([-0.05, -0.025, 0, 0.025, 0.05]), 3) if 0.05 <= x <= 0.75]
    search([(tp, tb) for tp in near(best[1][0]) for tb in near(best[1][1])])
    a, T = replay(*best[1])                                   # full chronological replay with the chosen thresholds
    return a, T, best[1]

In [ ]:
# R4 — supervised gradient boosting (GPU): decision model on the oracle, plus misuse scores for T2/T3
def xgb_model(seed, multi):
    p = dict(n_estimators=500, learning_rate=0.06, max_depth=8, subsample=0.9, colsample_bytree=0.8, tree_method="hist",
             device=DEVICE, enable_categorical=True, max_cat_to_onehot=1, random_state=seed)
    return xgb.XGBClassifier(objective="multi:softprob", **p) if multi else xgb.XGBClassifier(**p)

def gbdt(c, X, seed, train_mask=None):
    tr = c.tr if train_mask is None else train_mask
    cls = np.unique(c.opt[tr]); remap = np.searchsorted(cls, c.opt[tr])           # XGBoost needs contiguous labels
    m = xgb_model(seed, len(cls) > 2).fit(X[tr], remap); pp = m.predict_proba(X)
    pr = np.zeros((len(X), 4)); pr[:, cls] = pp if pp.shape[1] == len(cls) else np.c_[1 - pp[:, -1], pp[:, -1]]
    q = np.where(c.mask, pr, -1.0); a = q.argmax(1)                                 # masked arg-max: never outside the envelope
    s = xgb_model(seed, False).fit(X[tr], c.mal[tr].astype(int)).predict_proba(X)[:, 1]
    return a, pr, s

In [ ]:
# R5 — delayed-reward neural contextual bandit (GPU): learns only from the reward signal, rewards of deferred decisions arrive
#      with the revealed outcome (or a no-news reward after H days); chronological replay, masked actions
import torch.nn as nn
REW_NN, H_DAYS = 0.5, 21

def resolved_reward(a, label, z):
    return np.select([np.isin(label, ["Malicious", "DelegationMisused"]), label == "Legitimate", label == "CompromisedPath"],
                     [-5 * z * np.where(a == L, 0.5, 1.0), np.where(np.isin(a, [P, B]), 1.0, 0.5), np.where(a == L, 1.0, -z)], REW_NN)

def bandit(c, X, seed, passes=4, gamma_steps=4):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); d = c.d
    num = X.select_dtypes(include=[np.number]).fillna(0).to_numpy(np.float32)
    cat = pd.get_dummies(X.select_dtypes(exclude=[np.number]).astype(str), dtype=np.float32).to_numpy()
    F = np.c_[num, cat]; mu, sd = F[c.tr].mean(0), F[c.tr].std(0) + 1e-6; F = ((F - mu) / sd).astype(np.float32)
    Ft = torch.tensor(F, device=DEVICE); Mt = torch.tensor(c.mask, device=DEVICE)
    net = nn.Sequential(nn.Linear(F.shape[1], 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 4)).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), 3e-4)
    emerg = d.EmergencyFlag.to_numpy() == 1
    rid2row = pd.Series(np.arange(len(d)), index=d.RequestID.to_numpy())
    fb = c.ev.assign(row=rid2row[c.ev.RequestID.to_numpy()].to_numpy())
    fb = fb.assign(fday=((fb.FeedbackTime - d.Timestamp.min().normalize()).dt.days).to_numpy()).drop_duplicates("row", keep="last")
    # per-row lookup arrays (one revealed event per request at most; rows without an event never resolve before the horizon)
    has_fb = np.zeros(len(d), bool); has_fb[fb.row] = True
    fb_day = np.zeros(len(d), int); fb_day[fb.row] = fb.fday.to_numpy()
    fb_lab = np.full(len(d), "", object); fb_lab[fb.row] = fb.FeedbackLabel.to_numpy()
    fb_app = np.full(len(d), "always", object); fb_app[fb.row] = fb.AppliesIf.to_numpy()
    train_days = np.unique(c.day[c.tr]); by_day = {k: np.flatnonzero((c.day == k) & c.tr) for k in train_days}
    def q_all(rows):
        with torch.no_grad(): return torch.where(Mt[rows], net(Ft[rows]), torch.tensor(-1e9, device=DEVICE)).cpu().numpy()
    best = (-1, None)
    for ps in range(passes):
        eps = max(0.05, 1.0 - ps / max(passes - 1, 1))
        buf_s, buf_a, buf_r = [], [], []; pending = {}          # due_day -> [(rows, actions, labels)]
        for k in train_days:
            idx = by_day[k]
            q = q_all(idx); a = q.argmax(1)
            rnd = rng.random(len(idx)) < eps
            if rnd.any():                                       # uniform random admissible action (vectorised)
                a[rnd] = (rng.random((rnd.sum(), 4)) * c.mask[idx[rnd]]).argmax(1)
            defr = (a == B) | ((a == L) & emerg[idx]) | (c.grant[idx] & np.isin(a, [P, L]))
            r_imm = immediate_reward(a, c.opt[idx], c.mal[idx], c.zeta[idx])
            buf_s.append(idx[~defr]); buf_a.append(a[~defr]); buf_r.append(r_imm[~defr])
            if defr.any():
                di, da = idx[defr], a[defr]
                okf = has_fb[di] & (fb_day[di] <= k + H_DAYS) & applies(fb_app[di], da)
                due_day = np.where(okf, fb_day[di], k + H_DAYS); lab = np.where(okf, fb_lab[di], "")
                for dd in np.unique(due_day):
                    s_ = due_day == dd; pending.setdefault(int(dd), []).append((di[s_], da[s_], lab[s_]))
            due = [x for dd in sorted(dd for dd in pending if dd <= k) for x in pending.pop(dd)]
            if due:
                rr = np.concatenate([x[0] for x in due]); aa = np.concatenate([x[1] for x in due]); lab = np.concatenate([x[2] for x in due])
                buf_s.append(rr); buf_a.append(aa); buf_r.append(resolved_reward(aa, lab, c.zeta[rr]))
            S = np.concatenate(buf_s[-60:]); A = np.concatenate(buf_a[-60:]); R = np.concatenate(buf_r[-60:])
            for _ in range(gamma_steps):
                j = rng.integers(0, len(S), min(2048, len(S)))
                qv = net(Ft[S[j]]).gather(1, torch.tensor(A[j], device=DEVICE)[:, None]).squeeze(1)
                loss = nn.functional.smooth_l1_loss(qv, torch.tensor(R[j], dtype=torch.float32, device=DEVICE))
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 10); opt.step()
        av = q_all(np.flatnonzero(c.va)).argmax(1); a_full = np.full(len(d), D); a_full[c.va] = av
        f = val_reward(a_full, c)
        if f > best[0]: best = (f, {k: v.clone() for k, v in net.state_dict().items()})
    net.load_state_dict(best[1])
    q = q_all(np.arange(len(d)))
    return q.argmax(1), q

## Revision experiments (S0-S4)
New baselines (relationship-based access control, two more insider-threat detectors, a sequence model, an off-policy bandit),
10 model seeds with 95% confidence intervals, and an actor-disjoint protocol for all tasks. Every run is checkpointed as one
small JSON file in `reports/rev1/runs/`, so a disconnected runtime loses at most the run in progress: re-run R0-R5 and S0-S3.

In [ ]:
# S0 — settings for the revision runs
from scipy import stats
import torch.nn as nn
FAST = False          # True = quick smoke test (1 seed, main build, short training, separate folder); False = the real run
SEEDS = [0] if FAST else list(range(10))
PLAN = {"main": ["temporal", "actor_disjoint_all"], "hard_fraud": ["temporal", "actor_disjoint_all"],
        "sparse_feedback": ["temporal"], "noisy_benign": ["temporal"]}       # build -> protocols; delete entries to save time
PLAN = {b: p for b, p in PLAN.items() if b in RUN_BUILDS}
if FAST: PLAN = {"main": ["actor_disjoint_all"]}
EP = dict(ae=2, gru=1, off=2, passes=1) if FAST else dict(ae=8, gru=6, off=8, passes=4)
REV = REP / "rev1"; RUNS = REV / ("runs_fast" if FAST else "runs"); RUNS.mkdir(parents=True, exist_ok=True)
print("plan:", PLAN, "| seeds:", SEEDS, "| runs folder:", RUNS)

def actor_split(c):
    """Actor-disjoint split for every threat, derived from released columns only. For each threat type the malicious actors
    are sorted and every second one is held out: held-out actors' malicious rows appear only in the test period, the other
    actors' malicious rows only in training and validation. Actor = the device for device tampering, the delegatee for a
    delegation grant, the requesting user otherwise. For emergency misuse this equals the released `actor_disjoint` column.
    For the other threats, actors that are active in the test period and actors that are not are alternated separately,
    so that every threat keeps held-out actors with test-period requests (campaign threats are active only part of the time)."""
    d = c.d; th = d.ThreatType.to_numpy(object).copy()
    misg = c.grant & c.mis
    pos = c.mal | misg
    th[misg & ~c.mal] = "delegation_misuse"
    key = np.where(th == "virtual_object_tampering", d.DeviceID.to_numpy(object),
                   np.where(c.grant & d.DelegateeID.notna().to_numpy(), d.DelegateeID.to_numpy(object), d.UserID.to_numpy(object)))
    held = np.zeros(len(d), bool); info = {}
    for t in sorted(set(th[pos])):
        m = pos & (th == t); actors = sorted(set(key[m].tolist()))
        if t == "emergency_misuse": h = set(actors[1::2])
        else:
            late = set(key[m & (c.split == "test")].tolist())
            wt, wo = [x for x in actors if x in late], [x for x in actors if x not in late]
            h = set((wt[1::2] if len(wt) > 1 else wt) + wo[1::2])
        held |= m & pd.Series(key).isin(h).to_numpy()
        info[t] = f"{len(actors) - len(h)} seen / {len(h)} held out"
    sp = c.split.astype(object).copy()
    sp[pos & held & (c.split != "test")] = "excluded"
    sp[pos & ~held & (c.split == "test")] = "excluded"
    return sp, info

def set_protocol(c, proto):
    """temporal: the released split. actor_disjoint_all: see actor_split; feedback about excluded rows is removed as well,
    so that mechanisms which learn from revealed outcomes never see evidence about a held-out actor before the test period."""
    c.proto = proto; c.tem = c.te.copy(); c.info = {}
    if proto == "temporal": return
    sp, c.info = actor_split(c); excl = sp == "excluded"
    c.tr, c.va, c.tem = sp == "train", sp == "val", sp == "test"
    c.ev = c.ev[~c.ev.RequestID.isin(set(c.d.RequestID.to_numpy()[excl]))].reset_index(drop=True)

In [ ]:
# S1 — new baselines
def rebac(c):
    """Relationship-based access control: grant whenever a relationship links requester and resource (care team or department
    for clinicians, device department for technicians, consent for researchers, own record for patients, any delegation edge),
    without checking whether the role may perform the operation."""
    d = c.d; role = d.UserRole.to_numpy(object)
    rel = np.isin(role, ["Doctor", "Nurse"]) & ((d.OnCareTeam.to_numpy() == 1) | (d.SameDepartment.to_numpy() == 1))
    rel |= (role == "Technician") & (d.DeviceDepartment == d.UserDepartment).to_numpy()
    rel |= (role == "Administrator")
    rel |= (role == "Researcher") & (d.PatientResearchConsent.to_numpy() == 1)
    rel |= (role == "Patient") & c.mask[:, 0]
    rel |= d.DelegationID.notna().to_numpy()
    return np.where(c.grant, np.where(c.mask[:, 0], P, D), np.where(rel, P, D))

def user_deviation(c):
    """Insider detection by per-user behaviour deviation (label-free): mean absolute z-score of the history features against
    the user's own training-period profile (population profile for users with fewer than 30 training requests)."""
    H = c.H.fillna(0).to_numpy(np.float64); uid = pd.factorize(c.d.UserID)[0]; nU = uid.max() + 1; tr = c.tr
    cnt = np.bincount(uid[tr], minlength=nU).astype(float)[:, None]
    s1 = np.zeros((nU, H.shape[1])); np.add.at(s1, uid[tr], H[tr]); s2 = np.zeros_like(s1); np.add.at(s2, uid[tr], H[tr] ** 2)
    mu = s1 / np.maximum(cnt, 1); sd = np.sqrt(np.maximum(s2 / np.maximum(cnt, 1) - mu ** 2, 0))
    gmu, gsd = H[tr].mean(0), H[tr].std(0) + 1e-6
    few = cnt[:, 0] < 30; mu[few] = gmu; sd[few] = gsd
    z = np.abs(H - mu[uid]) / (sd[uid] + 0.25 * gsd)          # shrink towards the population spread
    s = np.clip(z, 0, 10).mean(1); lo, hi = np.quantile(s[tr], [0.01, 0.99])
    return np.clip((s - lo) / (hi - lo + 1e-9), 0, 1)

def feats(c, X):
    """standardised numeric columns + one-hot categorical columns (statistics from the training rows)"""
    num = X.select_dtypes(include=[np.number]).fillna(0).to_numpy(np.float32)
    cat = pd.get_dummies(X.select_dtypes(exclude=[np.number]).astype(str), dtype=np.float32).to_numpy()
    F = np.c_[num, cat]; mu, sd = F[c.tr].mean(0), F[c.tr].std(0) + 1e-6
    return np.clip((F - mu) / sd, -10, 10).astype(np.float32)

def autoencoder_score(c, seed, epochs=8):
    """Insider detection by reconstruction error (label-free): a small autoencoder fitted on training requests."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    num = c.X2.select_dtypes(include=[np.number]).fillna(0).to_numpy(np.float32)
    mu, sd = num[c.tr].mean(0), num[c.tr].std(0) + 1e-6; Z = np.clip((num - mu) / sd, -8, 8).astype(np.float32)
    Zt = torch.tensor(Z, device=DEVICE); k = Z.shape[1]
    net = nn.Sequential(nn.Linear(k, 64), nn.ReLU(), nn.Linear(64, 12), nn.ReLU(), nn.Linear(12, 64), nn.ReLU(), nn.Linear(64, k)).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), 1e-3); idx = np.flatnonzero(c.tr)
    for _ in range(epochs):
        rng.shuffle(idx)
        for i in range(0, len(idx), 2048):
            j = torch.tensor(idx[i:i + 2048], device=DEVICE)
            loss = ((net(Zt[j]) - Zt[j]) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        err = torch.cat([((net(Zt[i:i + 65536]) - Zt[i:i + 65536]) ** 2).mean(1) for i in range(0, len(Z), 65536)]).cpu().numpy()
    lo, hi = np.quantile(err[c.tr], [0.01, 0.99])
    return np.clip((err - lo) / (hi - lo + 1e-9), 0, 1)

def seq_index(c, K=16):
    """row indices of each request's K most recent requests by the same user (itself last); len(d) marks padding"""
    n = len(c.d); uid = pd.factorize(c.d.UserID)[0]
    order = np.lexsort((np.arange(n), uid)); us = uid[order]
    start = np.r_[0, np.flatnonzero(np.diff(us)) + 1]; rank = np.arange(n) - np.repeat(start, np.diff(np.r_[start, n]))
    idx = np.full((n, K), n, dtype=np.int64); pos = np.arange(n)
    for j in range(K):
        ok = rank >= j; idx[order[ok], K - 1 - j] = order[pos[ok] - j]
    return idx

class SeqNet(nn.Module):
    def __init__(self, k, h=128):
        super().__init__(); self.g = nn.GRU(k, h, batch_first=True)
        self.o = nn.Sequential(nn.Linear(h + k, 128), nn.ReLU(), nn.Linear(128, 4))
    def forward(self, x):
        out, _ = self.g(x); return self.o(torch.cat([out[:, -1], x[:, -1]], 1))

def gru_seq(c, seed, epochs=6, K=16):
    """Sequence model: a GRU over the observable attributes of the user's last K requests predicts the decision for the
    current one (trained on the oracle like gradient boosting; masked arg-max; epoch chosen by validation reward)."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    if getattr(c, "seq_idx", None) is None: c.seq_idx = seq_index(c, K)
    F = feats(c, c.X1); k = F.shape[1]
    Ft = torch.tensor(np.vstack([F, np.zeros((1, k), np.float32)]), device=DEVICE)        # last row = padding
    I = torch.tensor(c.seq_idx, device=DEVICE); y = torch.tensor(c.opt, dtype=torch.long, device=DEVICE)
    freq = np.bincount(c.opt[c.tr], minlength=4) + 1.0
    w = torch.tensor((freq.sum() / freq) ** 0.5, dtype=torch.float32, device=DEVICE); w = w / w.mean()
    net = SeqNet(k).to(DEVICE); opt = torch.optim.Adam(net.parameters(), 1e-3); tr = np.flatnonzero(c.tr); best = (-1e9, None, None)
    def predict():
        net.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(c.d), 8192):
                j = torch.arange(i, min(i + 8192, len(c.d)), device=DEVICE); out.append(torch.softmax(net(Ft[I[j]]), 1))
        net.train(); return torch.cat(out).cpu().numpy()
    for _ in range(epochs):
        rng.shuffle(tr)
        for i in range(0, len(tr), 1024):
            j = torch.tensor(tr[i:i + 1024], device=DEVICE)
            loss = nn.functional.cross_entropy(net(Ft[I[j]]), y[j], weight=w)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 5); opt.step()
        pr = predict(); a = np.where(c.mask, pr, -1.0).argmax(1); f = val_reward(a, c)
        if f > best[0]: best = (f, a, pr)
    return best[1], best[2]

def logged_feedback(c, seed, eps=0.1):
    """A log written by a fixed logging policy (static ABAC with static break-glass; with probability eps a uniformly random
    admissible action). Returns the logged action, its probability under the logging policy and the reward the log records
    (immediate, or for deferred decisions the revealed outcome if it applies to the action, otherwise the no-news reward)."""
    rng = np.random.default_rng(1000 + seed); d = c.d; a0 = static_bg(c); nA = c.mask.sum(1)
    rnd = rng.random(len(d)) < eps; a = a0.copy()
    a[rnd] = (rng.random((int(rnd.sum()), 4)) * c.mask[rnd]).argmax(1)
    p = np.where(a == a0, (1 - eps) + eps / nA, eps / nA)
    rid2row = pd.Series(np.arange(len(d)), index=d.RequestID.to_numpy())
    fb = c.ev.assign(row=rid2row[c.ev.RequestID.to_numpy()].to_numpy())
    fb = fb.assign(fday=((fb.FeedbackTime - d.Timestamp.min().normalize()).dt.days).to_numpy()).drop_duplicates("row", keep="last")
    has_fb = np.zeros(len(d), bool); has_fb[fb.row] = True
    fb_day = np.zeros(len(d), int); fb_day[fb.row] = fb.fday.to_numpy()
    fb_lab = np.full(len(d), "", object); fb_lab[fb.row] = fb.FeedbackLabel.to_numpy()
    fb_app = np.full(len(d), "always", object); fb_app[fb.row] = fb.AppliesIf.to_numpy()
    emerg = d.EmergencyFlag.to_numpy() == 1
    defr = (a == B) | ((a == L) & emerg) | (c.grant & np.isin(a, [P, L]))
    okf = has_fb & (fb_day <= c.day + H_DAYS) & applies(fb_app, a)
    r = immediate_reward(a, c.opt, c.mal, c.zeta)
    r = np.where(defr, resolved_reward(a, np.where(okf, fb_lab, ""), c.zeta), r)
    return a, p, r.astype(np.float32)

def off_policy(c, seed, epochs=8, eps=0.1, clip=20.0):
    """Off-policy contextual bandit (doubly robust): learns from the fixed log above, never acts during training. A reward
    model is fitted on the logged actions, corrected with clipped inverse-propensity weights, and a second network is fitted
    on the corrected targets for all admissible actions. The epoch is chosen by the doubly-robust value estimate on the
    validation part of the log (no oracle labels)."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    a_log, p, r = logged_feedback(c, seed, eps)
    F = feats(c, c.X2); Ft = torch.tensor(F, device=DEVICE); Mt = torch.tensor(c.mask, device=DEVICE)
    At = torch.tensor(a_log, dtype=torch.long, device=DEVICE); Rt = torch.tensor(r, device=DEVICE)
    Wt = torch.tensor(np.minimum(1 / p, clip).astype(np.float32), device=DEVICE)
    mlp = lambda: nn.Sequential(nn.Linear(F.shape[1], 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 4)).to(DEVICE)
    tr, va = np.flatnonzero(c.tr), torch.tensor(np.flatnonzero(c.va), device=DEVICE)
    def batches():
        rng.shuffle(tr)
        for i in range(0, len(tr), 2048): yield torch.tensor(tr[i:i + 2048], device=DEVICE)
    def full(net):
        with torch.no_grad(): return torch.cat([net(Ft[i:i + 65536]) for i in range(0, len(F), 65536)])
    q1 = mlp(); o1 = torch.optim.Adam(q1.parameters(), 1e-3)                    # 1) reward model on the logged actions
    for _ in range(epochs):
        for j in batches():
            loss = nn.functional.smooth_l1_loss(q1(Ft[j]).gather(1, At[j][:, None]).squeeze(1), Rt[j])
            o1.zero_grad(); loss.backward(); o1.step()
    Q = full(q1); ar = torch.arange(len(F), device=DEVICE)
    corr = Wt * (Rt - Q[ar, At])                                                # 2) doubly-robust correction on the logged action
    T = Q.clone(); T[ar, At] += corr
    q2 = mlp(); o2 = torch.optim.Adam(q2.parameters(), 3e-4); best = (-1e9, None)
    for _ in range(epochs):                                                     # 3) policy network on the corrected targets
        for j in batches():
            diff = nn.functional.smooth_l1_loss(q2(Ft[j]), T[j], reduction="none")
            loss = (diff * Mt[j]).sum() / Mt[j].sum()
            o2.zero_grad(); loss.backward(); o2.step()
        q = torch.where(Mt, full(q2), torch.tensor(-1e9, device=DEVICE)); pi = q.argmax(1)
        v = (Q[va, pi[va]] + (pi[va] == At[va]) * corr[va]).mean().item()     # doubly-robust value of the greedy policy
        if v > best[0]: best = (v, q.cpu().numpy())
    q = best[1]
    return q.argmax(1), q

def t2_ci(score, g, f, boot=1000):
    sg, sf = score[g], score[f]
    if len(sg) == 0 or len(sf) == 0: return {}
    out = {"AUROC": round(float(roc_auc_score(np.r_[np.ones(len(sg)), np.zeros(len(sf))], np.r_[sg, sf])), 4),
           "fraud granted @99%": round(float((sf >= np.quantile(sg, 0.01)).mean()), 4), "genuine": int(len(sg)), "fraud": int(len(sf))}
    if boot:
        rng = np.random.default_rng(0); bs = []
        for _ in range(boot):
            x, y_ = rng.choice(sg, len(sg)), rng.choice(sf, len(sf))
            bs.append(roc_auc_score(np.r_[np.ones(len(x)), np.zeros(len(y_))], np.r_[x, y_]))
        out["lo"], out["hi"] = [round(float(v), 4) for v in np.quantile(bs, [0.025, 0.975])]
    return out

def t2_detail(score, c, rows, boot):
    """T2 on emergency claims in break-glass contexts: genuine vs emergency-misuse fraud; overt / covert where both exist"""
    ctx = rows & c.mask[:, 3] & (c.d.EmergencyFlag.to_numpy() == 1)
    gen, fra = ctx & (c.opt == B), ctx & c.mal & (c.d.ThreatType.to_numpy() == "emergency_misuse")
    out = {"all": t2_ci(score, gen, fra, boot)}; tag = c.d.ActorTag.to_numpy()
    if (tag == "CovertEmergencyMisuser").any():
        out["overt"] = t2_ci(score, gen, fra & (tag == "EmergencyMisuser"), boot)
        out["covert"] = t2_ci(score, gen, fra & (tag == "CovertEmergencyMisuser"), boot)
    return out

def boot_rates(a, c, rows, nb=2000):
    """95% bootstrap interval over test requests for the three headline rates (exact for a proportion: binomial resampling)"""
    a, o, m = a[rows], c.opt[rows], c.mal[rows]; g = np.isin(a, [P, L, B]); rng = np.random.default_rng(0); out = {}
    for k, v in {"MGR": g[m], "LDR": a[o == P] == D, "EDR": a[o == B] == D}.items():
        if len(v):
            bs = rng.binomial(len(v), float(v.mean()), nb) / len(v)
            out[k] = [round(float(x), 4) for x in np.quantile(bs, [0.025, 0.975])]
    return out

In [ ]:
# S2 — run every approach under every protocol in PLAN (checkpointed per run)
slug = lambda s: "".join(ch if ch.isalnum() else "_" for ch in s)
run_file = lambda b, proto, name, sd: RUNS / f"{b}__{proto}__{slug(name)}__s{sd}.json"
DET = ["Static ABAC", "ABAC + static break-glass", "RBAC", "ReBAC (relationship-based)", "Mined ABAC (policy mining)", "CapBAC",
       "CapBAC + audit revocation", "Trust-based (beta reputation)", "Risk-adaptive", "Zero-trust re-evaluation",
       "Insider detection (Isolation Forest)", "Insider detection (user deviation)"]
LEARNED = ["Insider detection (autoencoder)", "Gradient boosting, single request", "Gradient boosting, with history",
           "Sequence model (GRU)", "Delayed-reward neural bandit", "Off-policy bandit (doubly robust)"]
ORDER = DET + LEARNED

def record(c, b, proto, name, sd, a, s2, s3):
    det = sd == "det"
    r = {"build": b, "proto": proto, "method": name, "seed": sd, "T1": t1_metrics(a, c, c.tem), "threat": per_threat(a, c, c.tem),
         "boot": boot_rates(a, c, c.tem), "n_test_malicious": int((c.tem & c.mal).sum()),
         "threat_n": {k: int(v) for k, v in pd.Series(c.d.ThreatType.to_numpy()[c.tem & c.mal]).value_counts().items()}}
    if s2 is not None: r["T2"] = t2_detail(np.asarray(s2, float), c, c.tem, 1000 if det else 0)
    if s3 is not None: r["T3"] = t3_metrics(np.asarray(s3, float), c, c.tem)
    json.dump(r, open(run_file(b, proto, name, sd), "w"), default=float)

def run_one(b, proto):
    todo_det = [n for n in DET if not run_file(b, proto, n, "det").exists()]
    todo = [(n, sd) for n in LEARNED for sd in SEEDS if not run_file(b, proto, n, sd).exists()]
    if not todo_det and not todo: print(f"{b} / {proto}: complete (loaded from checkpoints)"); return
    t0 = time.time(); c = Ctx(b); set_protocol(c, proto); c.seq_idx = None
    if c.info: print(f"{b} / {proto}: actors per threat -> {c.info}")
    if proto != "temporal" and (REL / b / "splits.parquet").exists():
        old = pd.read_parquet(REL / b / "splits.parquet")
        if "actor_disjoint" in old:
            em = (c.d.ThreatType == "emergency_misuse").to_numpy() & c.mal
            sp = np.select([c.tr, c.va, c.tem], ["train", "val", "test"], "excluded")
            print("   emergency-misuse rows identical to the released actor_disjoint split:", bool((sp[em] == old.actor_disjoint.to_numpy()[em]).all()))
    if todo_det:
        risk = an = None
        for n in todo_det:
            if n == "Static ABAC": out = (static_abac(c), None, None)
            elif n == "ABAC + static break-glass": out = (static_bg(c), None, None)
            elif n == "RBAC": out = (rbac(c), None, None)
            elif n == "ReBAC (relationship-based)": out = (rebac(c), None, None)
            elif n == "Mined ABAC (policy mining)": out = (mined_abac(c)[0], None, None)
            elif n == "CapBAC": out = (capbac(c, False), None, None)
            elif n == "CapBAC + audit revocation": out = (capbac(c, True), None, None)
            elif n == "Trust-based (beta reputation)": a, T, _ = trust_beta(c); out = (a, T, 1 - T)
            elif n == "Zero-trust re-evaluation": out = (zero_trust(c), None, None)
            elif n == "Insider detection (user deviation)": s = user_deviation(c); out = (isolation_insider(c, s), -s, s)
            else:
                if risk is None: a_risk, risk, an = risk_adaptive(c)
                out = (a_risk, -risk, risk) if n == "Risk-adaptive" else (isolation_insider(c, an), -an, an)
            record(c, b, proto, n, "det", *out)
        print(f"{b} / {proto}: rule-based and feedback approaches done ({time.time() - t0:.0f}s)")
    for n, sd in todo:
        if n == "Insider detection (autoencoder)": s = autoencoder_score(c, sd, EP["ae"]); out = (isolation_insider(c, s), -s, s)
        elif n.startswith("Gradient boosting"):
            a, pr, _ = gbdt(c, c.X1 if "single" in n else c.X2, sd); out = (a, pr[:, B] - pr[:, D], pr[:, D])
        elif n == "Sequence model (GRU)": a, pr = gru_seq(c, sd, EP["gru"]); out = (a, pr[:, B] - pr[:, D], pr[:, D])
        elif n == "Delayed-reward neural bandit":
            a, q = bandit(c, c.X2, sd, passes=EP["passes"]); out = (a, q[:, B] - q[:, D], q[:, D] - np.maximum(q[:, P], q[:, L]))
        else:
            a, q = off_policy(c, sd, EP["off"]); out = (a, q[:, B] - q[:, D], q[:, D] - np.maximum(q[:, P], q[:, L]))
        record(c, b, proto, n, sd, *out)
        print(f"{b} / {proto}: {n}, seed {sd} done ({time.time() - t0:.0f}s)", flush=True)
    del c; import gc; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

for b, protos in PLAN.items():
    for proto in protos: run_one(b, proto)

In [ ]:
# S3 — summary tables: mean over seeds with 95% confidence intervals (Student t over seeds for learned approaches,
#      bootstrap over test requests for deterministic ones)
def ci_seed(v):
    v = np.asarray([x for x in v if x is not None and not (isinstance(x, float) and np.isnan(x))], float)
    if len(v) == 0: return np.nan, np.nan, np.nan
    if len(v) < 2: return float(v.mean()), np.nan, np.nan
    h = stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v)); return float(v.mean()), float(v.mean() - h), float(v.mean() + h)
fmt = lambda m, lo, hi: "" if np.isnan(m) else (f"{m:.3f}" if np.isnan(lo) else f"{m:.3f} [{lo:.3f}, {hi:.3f}]")

R = [json.load(open(f)) for f in sorted(RUNS.glob("*.json"))]
G = {}
for r in R: G.setdefault((r["build"], r["proto"], r["method"]), []).append(r)
T1K = ["MacroF1", "MGR", "LDR", "EDR", "DG-Mis", "DU-Exp", "Avg reward"]
t1, th, t2, t3 = [], [], [], []
for (b, proto, m), rs in G.items():
    det = rs[0]["seed"] == "det"; row = {"build": b, "protocol": proto, "method": m, "seeds": len(rs)}
    for k in T1K:
        mean, lo, hi = ci_seed([r["T1"].get(k) for r in rs])
        if det and k in rs[0]["boot"]: lo, hi = rs[0]["boot"][k]
        row[k] = round(mean, 4); row[k + " 95% CI"] = "" if np.isnan(lo) else f"[{lo:.3f}, {hi:.3f}]"
    t1.append(row)
    keys = sorted({k for r in rs for k in r["threat"]})
    th.append({"build": b, "protocol": proto, "method": m, **{k: round(float(np.mean([r["threat"][k] for r in rs if k in r["threat"]])), 4) for k in keys}})
    for part in ["all", "overt", "covert"]:
        v = [r["T2"][part] for r in rs if "T2" in r and r["T2"].get(part)]
        if v:
            mean, lo, hi = ci_seed([x["AUROC"] for x in v])
            if det and "lo" in v[0]: lo, hi = v[0]["lo"], v[0]["hi"]
            t2.append({"build": b, "protocol": proto, "method": m, "fraudsters": part, "T2 AUROC": fmt(mean, lo, hi),
                       "fraud granted @99%": round(float(np.mean([x["fraud granted @99%"] for x in v])), 3),
                       "genuine": v[0]["genuine"], "fraud": v[0]["fraud"], "seeds": len(v)})
    v = [r["T3"] for r in rs if r.get("T3")]
    if v:
        t3.append({"build": b, "protocol": proto, "method": m, "T3 AUROC": fmt(*ci_seed([x["T3 AUROC"] for x in v])),
                   "T3 AUPRC": fmt(*ci_seed([x["T3 AUPRC"] for x in v])), "base rate": v[0]["T3 base rate"], "seeds": len(v)})
key = lambda t: t.assign(_o=t.method.map({m: i for i, m in enumerate(ORDER)})).sort_values(["build", "protocol", "_o"]).drop(columns="_o")
T1 = key(pd.DataFrame(t1)); TH = key(pd.DataFrame(th)); T2 = key(pd.DataFrame(t2)) if t2 else pd.DataFrame(); T3 = key(pd.DataFrame(t3)) if t3 else pd.DataFrame()
T1.to_csv(REV / "T1_summary.csv", index=False); TH.to_csv(REV / "per_threat_MGR.csv", index=False)
T2.to_csv(REV / "T2_summary.csv", index=False); T3.to_csv(REV / "T3_summary.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
for (b, proto), t in T1.groupby(["build", "protocol"], sort=False):
    print(f"\n=== T1 | {b} | {proto} ==="); print(t.drop(columns=["build", "protocol"]).to_string(index=False))
if len(T2): print("\n=== T2 (emergency claims; genuine vs fraudulent) ==="); print(T2.to_string(index=False))
if len(T3): print("\n=== T3 (delegation risk) ==="); print(T3.to_string(index=False))
# seen vs held-out actors: how much of each approach's result depends on having seen the actors
NT = pd.DataFrame([{"build": b, "protocol": proto, **rs[0].get("threat_n", {})} for (b, proto, m), rs in G.items() if m == "Static ABAC"])
NT.to_csv(REV / "per_threat_test_counts.csv", index=False); print("\n=== malicious test requests per threat ==="); print(NT.to_string(index=False))
piv = T1.pivot_table(index=["build", "method"], columns="protocol", values=["MGR", "MacroF1"], sort=False)
if piv.columns.get_level_values(1).nunique() > 1:
    piv.round(3).to_csv(REV / "actor_gap.csv"); print("\n=== temporal vs actor-disjoint (all threats) ==="); print(piv.round(3).to_string())

In [ ]:
# S4 — per-threat malicious-grant rate under both protocols (CSV for the paper) and the heatmap of Fig. 3 with the new baselines
import matplotlib.pyplot as plt
THREAT_NAME = {"gradual_decay": "gradual\nsnooping", "bursty_attack": "account\ntakeover", "collusion": "collusion",
               "delegation_misuse": "delegation\nmisuse", "privilege_escalation": "privilege\nescalation",
               "emergency_misuse": "emergency\nmisuse", "virtual_object_tampering": "device\ntampering"}
for proto, fn in [("temporal", "fig3_per_threat.pdf"), ("actor_disjoint_all", "fig3_per_threat_actor_disjoint.pdf")]:
    t = TH[(TH.build == "main") & (TH.protocol == proto)].set_index("method").reindex([m for m in ORDER if m in set(TH.method)])
    cols = [c_ for c_ in THREAT_NAME if c_ in t.columns]; hm = t[cols].dropna(how="all")
    if hm.empty: continue
    fig, ax = plt.subplots(figsize=(11, 0.55 * len(hm) + 2))
    im = ax.imshow(hm.to_numpy(float), cmap="Reds", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([THREAT_NAME[c_] for c_ in cols], fontsize=11)
    ax.xaxis.tick_top(); ax.xaxis.set_label_position("top"); ax.set_xlabel("Threat type", fontsize=12, labelpad=10)
    ax.set_yticks(range(len(hm))); ax.set_yticklabels(hm.index, fontsize=11); ax.set_ylabel("Reference baseline", fontsize=12)
    for i in range(hm.shape[0]):
        for j in range(hm.shape[1]):
            v = hm.iat[i, j]
            if not np.isnan(v): ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10.5, color="white" if v > 0.6 else "black")
    lab = "test, main build" if proto == "temporal" else "held-out actors, main build"
    cb = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02); cb.set_label(f"Malicious-grant rate ({lab})", fontsize=12)
    fig.tight_layout(); fig.savefig(REV / fn, bbox_inches="tight"); plt.show()
print("saved to", REV)